<a href="https://colab.research.google.com/github/StarmapC/bangla-english-nlp-study/blob/main/Research_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# FULL MULTILINGUAL NLP PIPELINE
# Bangla-English Code-Mixed Social Media Dataset
# ==========================================================

!pip -q install bertopic sentence-transformers transformers torch openpyxl nltk statsmodels

import pandas as pd
import numpy as np
import re
import nltk
import torch
from datetime import datetime, timezone
from nltk.corpus import stopwords
from scipy.stats import skew, shapiro, normaltest
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

nltk.download("stopwords")

# ----------------------------------------------------------
# 0. LOAD DATA
# ----------------------------------------------------------
file_path = "/content/dataset_zidan-research_2026-02-15_10-06-48-661 (3) (1).json"
df_raw = pd.read_json(file_path)

print("RAW DATASET SHAPE:", df_raw.shape)


# ----------------------------------------------------------
# 1. REMOVE EMPTY / MISSING POSTS FIRST (LOCK DATA QUALITY)
# ----------------------------------------------------------
df = df_raw.copy()

# Ensure required columns exist
required_cols = ["postId", "text", "time", "likes", "comments", "shares"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns in JSON: {missing_required}")

# Remove rows where text is missing/empty/only spaces
df["text"] = df["text"].astype(str)
df["text"] = df["text"].replace("nan", "").replace("None", "")

df = df[df["text"].str.strip() != ""].copy()

# Remove rows where likes/comments/shares are missing
df = df[
    df["likes"].notna() &
    df["comments"].notna() &
    df["shares"].notna()
].copy()

# Convert numeric safely
df["likes"] = pd.to_numeric(df["likes"], errors="coerce")
df["comments"] = pd.to_numeric(df["comments"], errors="coerce")
df["shares"] = pd.to_numeric(df["shares"], errors="coerce")

# Drop rows that became NaN after conversion
df = df.dropna(subset=["likes", "comments", "shares"]).copy()

# Remove negative engagement if exists (should not exist but safe)
df = df[(df["likes"] >= 0) & (df["comments"] >= 0) & (df["shares"] >= 0)].copy()

print("AFTER INITIAL CLEANING SHAPE:", df.shape)
print("N after cleaning =", len(df))


# ----------------------------------------------------------
# 2. RAW TOTAL ENGAGEMENT
# ----------------------------------------------------------
df["raw_total_engagement"] = df["likes"] + df["comments"] + df["shares"]


# ----------------------------------------------------------
# 3. SKEWNESS TEST (RAW ENGAGEMENT)
# ----------------------------------------------------------
raw_skew = skew(df["raw_total_engagement"])

print("\n==============================")
print("RAW ENGAGEMENT DISTRIBUTION")
print("==============================")
print("N =", len(df))
print("RAW SKEWNESS =", raw_skew)
print("MEAN =", df["raw_total_engagement"].mean())
print("MEDIAN =", df["raw_total_engagement"].median())
print("STD =", df["raw_total_engagement"].std())

# Normality tests (note: large N will almost always reject normality)
shapiro_result = shapiro(df["raw_total_engagement"].sample(min(5000, len(df))))
dagostino_result = normaltest(df["raw_total_engagement"])

print("\nShapiro-Wilk Test (sampled if large N):", shapiro_result)
print("D’Agostino-Pearson Test:", dagostino_result)


# ----------------------------------------------------------
# 4. LOG NORMALIZATION (IF SKEWNESS HIGH)
# ----------------------------------------------------------
df["log_raw_engagement"] = np.log1p(df["raw_total_engagement"])

log_skew = skew(df["log_raw_engagement"])

print("\n==============================")
print("LOG NORMALIZATION RESULT")
print("==============================")
print("ORIGINAL SKEWNESS =", raw_skew)
print("LOG SKEWNESS =", log_skew)


# ----------------------------------------------------------
# 5. TIME NORMALIZED ENGAGEMENT (USING RAW ENGAGEMENT)
# ----------------------------------------------------------
df["post_time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["post_time"]).copy()

now = datetime.now(timezone.utc)

df["post_age_days"] = ((now - df["post_time"]).dt.total_seconds() / 86400).fillna(0)

df["time_normalized_engagement"] = (
    df["raw_total_engagement"] / (df["post_age_days"] + 1)
)

print("\nN after time conversion =", len(df))


# ----------------------------------------------------------
# 6. CHECK DEGREE OF INDEPENDENCE (AUTOCORRELATION TEST)
# ----------------------------------------------------------
# Sort by time to check temporal dependency
df = df.sort_values("post_time").copy()

# Durbin-Watson test (closer to 2 = independence)
dw_stat = durbin_watson(df["time_normalized_engagement"])
print("\n==============================")
print("INDEPENDENCE CHECK")
print("==============================")
print("Durbin-Watson Statistic =", dw_stat)
print("Interpretation: ~2 = independent, <2 positive autocorrelation, >2 negative autocorrelation")


# ----------------------------------------------------------
# 7. NLP PREPROCESSING (MULTILINGUAL SAFE)
# ----------------------------------------------------------

# English stopwords (automatically loaded list)
stop_words_en = set(stopwords.words("english"))

# Show example of English stopwords (for transparency)
print("\nExample English stopwords (first 30):", list(stop_words_en)[:30])

# Bangla stopwords (explicit list; can be expanded)
bangla_stopwords = set([
    "এবং","ও","কিন্তু","যদি","তবে","কারণ","যে","যার","তার","এটা","এটি",
    "এই","সেই","ওই","আমি","আমরা","তুমি","আপনি","তারা","তিনি","কি","কেন",
    "কিভাবে","হয়","হবে","না","আর","একটি","একটা","গুলো","গুলি","দিয়ে","থেকে",
    "পর","মধ্যে","উপর","নিচে","সাথে","সব","অনেক","কিছু","যখন","তখন","যেখানে","সেখানে"
])

# contraction expansion (English minimal safe set)
contractions = {
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "doesn't": "does not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "i'm": "i am",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "they're": "they are",
    "you're": "you are",
    "we're": "we are"
}

def expand_contractions(text):
    for c, full in contractions.items():
        text = re.sub(r"\b" + re.escape(c) + r"\b", full, text)
    return text

def preprocess_text_multilingual(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # lowercase
    text = text.lower()

    # expand contractions (English only)
    text = expand_contractions(text)

    # remove urls
    text = re.sub(r"http\S+|www\S+", " ", text)

    # remove mentions
    text = re.sub(r"@\w+", " ", text)

    # remove hashtags symbol but keep word
    text = text.replace("#", " ")

    # remove emojis and non-text symbols (keep Bangla + English only)
    text = re.sub(r"[^a-zA-Z\u0980-\u09FF\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # tokenization
    tokens = text.split()

    # stopword removal
    tokens = [
        t for t in tokens
        if t not in stop_words_en
        and t not in bangla_stopwords
        and len(t) > 1
    ]

    return " ".join(tokens)

df["final_text"] = df["text"].apply(preprocess_text_multilingual)

# Remove rows that became empty after preprocessing
df = df[df["final_text"].str.strip() != ""].copy()

print("\nN after NLP preprocessing =", len(df))


# ----------------------------------------------------------
# 8. CODE SWITCHING METRICS (CMI)
# Das & Gambäck (2014) formula implemented
# ----------------------------------------------------------

def detect_token_lang(token):
    if re.search(r"[\u0980-\u09FF]", token):
        return "bn"
    elif re.search(r"[a-zA-Z]", token):
        return "en"
    else:
        return "other"

def compute_cmi(text):
    tokens = text.split()

    if len(tokens) == 0:
        return 0

    lang_tags = [detect_token_lang(t) for t in tokens]

    n = len(tokens)
    u = sum(1 for x in lang_tags if x == "other")

    if n == u:
        return 0

    bn_count = sum(1 for x in lang_tags if x == "bn")
    en_count = sum(1 for x in lang_tags if x == "en")

    max_wi = max(bn_count, en_count)

    cmi = 100 * (1 - (max_wi / (n - u)))
    return cmi

def classify_language_type(text):
    tokens = text.split()
    if len(tokens) == 0:
        return "non_linguistic"

    bn_count = sum(1 for t in tokens if re.search(r"[\u0980-\u09FF]", t))
    en_count = sum(1 for t in tokens if re.search(r"[a-zA-Z]", t))

    if bn_count > 0 and en_count == 0:
        return "pure_bangla"
    elif en_count > 0 and bn_count == 0:
        return "pure_english"
    elif bn_count > 0 and en_count > 0:
        return "code_mixed"
    else:
        return "non_linguistic"

df["CMI"] = df["final_text"].apply(compute_cmi)
df["language_type"] = df["final_text"].apply(classify_language_type)


# ----------------------------------------------------------
# 9. SENTIMENT ANALYSIS (MULTILINGUAL MODEL)
# Efficient batch prediction (prevents Colab crash)
# ----------------------------------------------------------

device = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device
)

def batch_sentiment(texts, batch_size=16):
    labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        results = sentiment_pipe(batch, truncation=True, max_length=512)
        labels.extend([r["label"].lower() for r in results])
    return labels

df["sentiment"] = batch_sentiment(df["final_text"].tolist(), batch_size=16)

print("\nSentiment completed.")


# ----------------------------------------------------------
# 10. TOPIC MODELING (BERTopic)
# ----------------------------------------------------------

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=False
)

topics, _ = topic_model.fit_transform(df["final_text"].tolist())
df["topic_id"] = topics

# Extract topic keywords
topic_keywords = {}
unique_topics = set(topics)

for t in unique_topics:
    if t == -1:
        continue
    words = topic_model.get_topic(t)
    if words:
        topic_keywords[t] = ", ".join([w[0] for w in words[:6]])

df["topic_keywords"] = df["topic_id"].map(topic_keywords)
df["topic_keywords"] = df["topic_keywords"].fillna("outlier/unclear")

print("\nTopic modeling completed.")


# ----------------------------------------------------------
# 11. FINAL LOCKED DATASET (EXPORT READY)
# ----------------------------------------------------------

df_final = df[[
    "postId",
    "final_text",
    "topic_id",
    "topic_keywords",
    "sentiment",
    "CMI",
    "language_type",
    "time_normalized_engagement"
]].copy()

print("\n==============================")
print("FINAL LOCKED DATASET READY")
print("==============================")
print("FINAL N =", len(df_final))
print("FINAL COLUMNS =", df_final.shape[1])

# ----------------------------------------------------------
# 12. EXPORT TO EXCEL
# ----------------------------------------------------------
output_path = "/content/final_research_dataset.xlsx"
df_final.to_excel(output_path, index=False)

print("\nEXCEL SAVED:", output_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.1 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


RAW DATASET SHAPE: (713, 38)
AFTER INITIAL CLEANING SHAPE: (614, 38)
N after cleaning = 614

RAW ENGAGEMENT DISTRIBUTION
N = 614
RAW SKEWNESS = 10.757261815980698
MEAN = 1630.5895765472312
MEDIAN = 354.5
STD = 5876.374285630361

Shapiro-Wilk Test (sampled if large N): ShapiroResult(statistic=np.float64(0.2225860538902844), pvalue=np.float64(1.4985807343732618e-44))
D’Agostino-Pearson Test: NormaltestResult(statistic=np.float64(1053.5702793288533), pvalue=np.float64(1.6600481885760222e-229))

LOG NORMALIZATION RESULT
ORIGINAL SKEWNESS = 10.757261815980698
LOG SKEWNESS = 0.14187020563101505

N after time conversion = 614

INDEPENDENCE CHECK
Durbin-Watson Statistic = 1.922127047585604
Interpretation: ~2 = independent, <2 positive autocorrelation, >2 negative autocorrelation

Example English stopwords (first 30): ['both', 'couldn', "should've", 'should', "it'll", 'mightn', 'between', 'about', 'but', "she's", 'will', 'then', "hasn't", 'm', 'most', 'can', 're', 'what', 'don', 'only', 'our', 

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Topic modeling completed.

FINAL LOCKED DATASET READY
FINAL N = 614
FINAL COLUMNS = 8

EXCEL SAVED: /content/final_research_dataset.xlsx


In [2]:
import numpy as np
from scipy.stats import skew

# Create log-transformed engagement column
df_final["log_time_engagement"] = np.log1p(df_final["time_normalized_engagement"])

# Check skewness
print("Skewness of time_normalized_engagement:", skew(df_final["time_normalized_engagement"]))
print("Skewness of log_time_engagement:", skew(df_final["log_time_engagement"]))

# Export updated dataset to Excel
output_path = "/content/final_research_dataset_with_logTNE.xlsx"
df_final.to_excel(output_path, index=False)

print("Updated Excel saved at:", output_path)

Skewness of time_normalized_engagement: 10.92984073632127
Skewness of log_time_engagement: 1.1020807412742417
Updated Excel saved at: /content/final_research_dataset_with_logTNE.xlsx


In [3]:
from sklearn.preprocessing import PowerTransformer
from scipy.stats import skew
import pandas as pd
import numpy as np

# Copy
df_final2 = df_final.copy()

# Prepare data
X = df_final2[["time_normalized_engagement"]].values

# Yeo-Johnson Transform
pt = PowerTransformer(method="yeo-johnson", standardize=True)
df_final2["yj_time_engagement"] = pt.fit_transform(X)

# Skewness check
print("Original TNE skewness:", skew(df_final2["time_normalized_engagement"]))
print("Log(TNE) skewness:", skew(np.log1p(df_final2["time_normalized_engagement"])))
print("Yeo-Johnson skewness:", skew(df_final2["yj_time_engagement"]))

# Export updated file
output_path = "/content/final_research_dataset_with_YJ.xlsx"
df_final2.to_excel(output_path, index=False)

print("Saved Excel:", output_path)

Original TNE skewness: 10.92984073632127
Log(TNE) skewness: 1.1020807412742417
Yeo-Johnson skewness: 0.11941784340697756
Saved Excel: /content/final_research_dataset_with_YJ.xlsx


In [5]:
import pandas as pd
import numpy as np
from scipy.stats import f_oneway, pearsonr
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# df_final2 contains yj_time_engagement (from Yeo-Johnson step)
df_stat = df_final2.copy()

print("FINAL N USED FOR STATISTICS:", len(df_stat))


# ==========================================================
# SUMMARY TABLE FUNCTION (USING YJ TRANSFORMED ENGAGEMENT)
# ==========================================================
def summary_table(group_col):
    summary = df_stat.groupby(group_col)["yj_time_engagement"].agg(
        N="count",
        Mean="mean",
        Median="median",
        Std="std"
    ).sort_values("Median", ascending=False)
    return summary


# ==========================================================
# 1) TOPIC_ID vs YJ ENGAGEMENT (ANOVA + TUKEY)
# ==========================================================
print("\n==============================")
print("1) TOPIC_ID vs YJ ENGAGEMENT")
print("==============================")

topic_summary = summary_table("topic_id")
display(topic_summary)

topic_groups = [
    group["yj_time_engagement"].values
    for name, group in df_stat.groupby("topic_id")
]

anova_topic = f_oneway(*topic_groups)

print("\nANOVA RESULT (Topic):")
print("F-statistic =", anova_topic.statistic)
print("p-value     =", anova_topic.pvalue)

tukey_topic = pairwise_tukeyhsd(
    endog=df_stat["yj_time_engagement"],
    groups=df_stat["topic_id"],
    alpha=0.05
)

tukey_topic_df = pd.DataFrame(
    data=tukey_topic.summary().data[1:],
    columns=tukey_topic.summary().data[0]
)

print("\nTukey HSD Posthoc (Topic):")
display(tukey_topic_df)


# ==========================================================
# 2) SENTIMENT vs YJ ENGAGEMENT (ANOVA + TUKEY)
# ==========================================================
print("\n==============================")
print("2) SENTIMENT vs YJ ENGAGEMENT")
print("==============================")

sent_summary = summary_table("sentiment")
display(sent_summary)

sent_groups = [
    group["yj_time_engagement"].values
    for name, group in df_stat.groupby("sentiment")
]

anova_sent = f_oneway(*sent_groups)

print("\nANOVA RESULT (Sentiment):")
print("F-statistic =", anova_sent.statistic)
print("p-value     =", anova_sent.pvalue)

tukey_sent = pairwise_tukeyhsd(
    endog=df_stat["yj_time_engagement"],
    groups=df_stat["sentiment"],
    alpha=0.05
)

tukey_sent_df = pd.DataFrame(
    data=tukey_sent.summary().data[1:],
    columns=tukey_sent.summary().data[0]
)

print("\nTukey HSD Posthoc (Sentiment):")
display(tukey_sent_df)


# ==========================================================
# 3) LANGUAGE_TYPE vs YJ ENGAGEMENT (ANOVA + TUKEY)
# ==========================================================
print("\n==============================")
print("3) LANGUAGE_TYPE vs YJ ENGAGEMENT")
print("==============================")

lang_summary = summary_table("language_type")
display(lang_summary)

lang_groups = [
    group["yj_time_engagement"].values
    for name, group in df_stat.groupby("language_type")
]

anova_lang = f_oneway(*lang_groups)

print("\nANOVA RESULT (Language Type):")
print("F-statistic =", anova_lang.statistic)
print("p-value     =", anova_lang.pvalue)

tukey_lang = pairwise_tukeyhsd(
    endog=df_stat["yj_time_engagement"],
    groups=df_stat["language_type"],
    alpha=0.05
)

tukey_lang_df = pd.DataFrame(
    data=tukey_lang.summary().data[1:],
    columns=tukey_lang.summary().data[0]
)

print("\nTukey HSD Posthoc (Language Type):")
display(tukey_lang_df)


# ==========================================================
# 4) CMI vs YJ ENGAGEMENT (PEARSON CORRELATION)
# ==========================================================
print("\n==============================")
print("4) CMI vs YJ ENGAGEMENT (Pearson Correlation)")
print("==============================")

pearson_cmi = pearsonr(df_stat["CMI"], df_stat["yj_time_engagement"])

print("Pearson r =", pearson_cmi.statistic)
print("p-value   =", pearson_cmi.pvalue)

pearson_df = pd.DataFrame({
    "Test": ["Pearson Correlation"],
    "Variable_X": ["CMI"],
    "Variable_Y": ["yj_time_engagement"],
    "r_value": [pearson_cmi.statistic],
    "p_value": [pearson_cmi.pvalue]
})


# ==========================================================
# BEST PERFORMING TOPIC + LANGUAGE TYPE (RANKED)
# ==========================================================
print("\n==============================")
print("TOP TOPICS BY MEDIAN (YJ Engagement)")
print("==============================")
display(topic_summary.head(10))

print("\n==============================")
print("LANGUAGE TYPE RANKING BY MEDIAN (YJ Engagement)")
print("==============================")
display(lang_summary)


# ==========================================================
# EXPORT RESULTS TO EXCEL
# ==========================================================
output_stats_path = "/content/statistical_results_FINAL_YJ.xlsx"

anova_results_df = pd.DataFrame({
    "Comparison": ["Topic vs Engagement", "Sentiment vs Engagement", "Language Type vs Engagement"],
    "F_statistic": [anova_topic.statistic, anova_sent.statistic, anova_lang.statistic],
    "p_value": [anova_topic.pvalue, anova_sent.pvalue, anova_lang.pvalue]
})

with pd.ExcelWriter(output_stats_path, engine="openpyxl") as writer:
    topic_summary.to_excel(writer, sheet_name="Topic_Summary_YJ")
    tukey_topic_df.to_excel(writer, sheet_name="Topic_Tukey", index=False)

    sent_summary.to_excel(writer, sheet_name="Sentiment_Summary_YJ")
    tukey_sent_df.to_excel(writer, sheet_name="Sentiment_Tukey", index=False)

    lang_summary.to_excel(writer, sheet_name="Language_Summary_YJ")
    tukey_lang_df.to_excel(writer, sheet_name="Language_Tukey", index=False)

    pearson_df.to_excel(writer, sheet_name="CMI_Pearson", index=False)
    anova_results_df.to_excel(writer, sheet_name="ANOVA_Results", index=False)

print("\nEXCEL FILE EXPORTED:", output_stats_path)

FINAL N USED FOR STATISTICS: 614

1) TOPIC_ID vs YJ ENGAGEMENT


,N,Mean,Median,Std
topic_id,,,,
10,14,0.856109,1.114093,0.798194
12,12,0.566966,0.507615,0.320547
0,130,0.497395,0.450692,0.905245
2,79,0.315397,0.351446,1.032467
11,13,0.247289,0.251893,0.578079
5,33,0.004877,-0.114360,0.531354
1,81,-0.008849,-0.158191,0.976584
6,23,0.200490,-0.177420,0.718101
8,18,-0.228041,-0.277006,0.832222



ANOVA RESULT (Topic):
F-statistic = 16.44141994615765
p-value     = 2.622896424421028e-34

Tukey HSD Posthoc (Topic):


,group1,group2,meandiff,p-adj,lower,upper,reject
0,-1,0,0.7327,0.0000,0.2829,1.1826,True
1,-1,1,0.2265,0.9650,-0.2658,0.7188,False
2,-1,2,0.5507,0.0138,0.0557,1.0457,True
3,-1,3,-1.1541,0.0000,-1.6976,-0.6107,True
4,-1,4,0.0196,1.0000,-0.5386,0.5778,False
...,...,...,...,...,...,...,...
100,10,12,-0.2891,0.9999,-1.4420,0.8637,False
101,10,13,-0.9512,0.2814,-2.1320,0.2296,False
102,11,12,0.3197,0.9998,-0.8535,1.4928,False
103,11,13,-0.3424,0.9997,-1.5430,0.8582,False



2) SENTIMENT vs YJ ENGAGEMENT


...


4) CMI vs YJ ENGAGEMENT (Pearson Correlation)
Pearson r = -0.006096226980708578
p-value   = 0.8801712464183128

TOP TOPICS BY MEDIAN (YJ Engagement)


,N,Mean,Median,Std
topic_id,,,,
10,14,0.856109,1.114093,0.798194
12,12,0.566966,0.507615,0.320547
0,130,0.497395,0.450692,0.905245
2,79,0.315397,0.351446,1.032467
11,13,0.247289,0.251893,0.578079
5,33,0.004877,-0.114360,0.531354
1,81,-0.008849,-0.158191,0.976584
6,23,0.200490,-0.177420,0.718101
8,18,-0.228041,-0.277006,0.832222



LANGUAGE TYPE RANKING BY MEDIAN (YJ Engagement)


,N,Mean,Median,Std
language_type,,,,
pure_bangla,102,0.503810,0.441373,0.945038
code_mixed,331,0.070231,-0.047426,0.924654
pure_english,181,-0.412348,-0.466035,1.009274



EXCEL FILE EXPORTED: /content/statistical_results_FINAL_YJ.xlsx


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
